In [ ]:
from google.colab import files
import pandas as pd
from sklearn.preprocessing import LabelEncoder

uploaded = files.upload()

file_name = list(uploaded.keys())[0]
data = pd.read_csv(file_name)

data = data.drop("Student_ID", axis=1)

le = LabelEncoder()

for column in ['Gender', 'Branch', 'Programming_Skills', 'Aptitude_Score', 'Communication_Skills', 'Placement_Status']:
    data[column] = le.fit_transform(data[column])

print("Data Loaded Successfully")

Saving AIDataset.csv to AIDataset.csv
Data Loaded Successfully


In [ ]:
from sklearn.model_selection import train_test_split

X = data.drop('Placement_Status', axis=1)
y = data['Placement_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

model = RandomForestClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Base Accuracy:", accuracy_score(y_test, y_pred))

Base Accuracy: 0.940677966101695


In [ ]:
import numpy as np
import random
from sklearn.model_selection import cross_val_score

def fitness_function(mask):
    selected = X.columns[mask == 1]

    if len(selected) == 0:
        return 0

    X_sel = X[selected]

    model = RandomForestClassifier()
    scores = cross_val_score(model, X_sel, y, cv=3)

    return scores.mean()

num_features = X.shape[1]
population_size = 10

population = [np.random.randint(0, 2, num_features) for _ in range(population_size)]

for gen in range(5):
    print("Generation:", gen)

    scored = [(ind, fitness_function(ind)) for ind in population]
    scored.sort(key=lambda x: x[1], reverse=True)

    print("Best score:", scored[0][1])

    top = [x[0] for x in scored[:5]]
    new_pop = top.copy()

    while len(new_pop) < population_size:
        p1, p2 = random.choice(top), random.choice(top)

        cut = random.randint(1, num_features-1)
        child = np.concatenate((p1[:cut], p2[cut:]))

        mut = random.randint(0, num_features-1)
        child[mut] = 1 - child[mut]

        new_pop.append(child)

    population = new_pop

best = scored[0][0]
selected_features = X.columns[best == 1]

print("Selected Features:", list(selected_features))

Generation: 0
Best score: 0.8464800678541137
Generation: 1
Best score: 0.847328244274809
Generation: 2
Best score: 0.9448685326547922
Generation: 3
Best score: 0.9414758269720102
Generation: 4
Best score: 0.9414758269720102
Selected Features: ['CGPA', 'Internships', 'Backlogs', 'Programming_Skills', 'Aptitude_Score', 'Communication_Skills', 'Extra_Certifications']


In [ ]:

from sklearn.model_selection import train_test_split

X_best = X[selected_features]

X_train, X_test, y_train, y_test = train_test_split(X_best, y, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("GA Accuracy:", accuracy_score(y_test, y_pred))

GA Accuracy: 0.961864406779661


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

salary_col = None
if "Salary_Offered_USD" in data.columns:
    salary_col = "Salary_Offered_USD"

if salary_col:
    print("Salary column found:", salary_col)

    y_reg = data[salary_col]

    # remove rows where salary is missing (just in case)
    valid_idx = y_reg.notna()
    X_reg = X_best[valid_idx]
    y_reg = y_reg[valid_idx]

    X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
        X_reg, y_reg, test_size=0.2, random_state=42
    )

    reg = RandomForestRegressor(random_state=42)
    reg.fit(X_train_r, y_train_r)

    print("Salary model trained ✔")

else:
    reg = None
    print("No salary column found")

Salary column found: Salary_Offered_USD
Salary model trained ✔


In [ ]:
class SimpleRL:
    def __init__(self):
        self.weights = {f: 1.0 for f in selected_features}
        self.lr = 0.05

    def adjust(self, row):
        return [row[i] * self.weights[selected_features[i]] for i in range(len(row))]

    def update(self, row, feedback):
        direction = 1 if feedback == "correct" else -1

        for i, f in enumerate(selected_features):
            self.weights[f] += direction * self.lr * row[i]

rl = SimpleRL()

In [ ]:
import pandas as pd

def predict_student(input_data):

    adjusted = rl.adjust(input_data)

    student = pd.DataFrame([adjusted], columns=selected_features)

    prob = model.predict_proba(student)[0][1]

    print("\n🎯 Prediction Result")
    print("Placement Probability:", round(prob, 2))

    if prob >= 0.5:
        print("Prediction: PLACED 🎉")

        if reg:
            salary = reg.predict(student)[0]
            print("Expected Salary:", round(salary, 2))
    else:
        print("Prediction: NOT PLACED ❌")
        print("Expected Salary: 0")

    feedback = input("Was this correct? (correct/wrong): ")
    rl.update(input_data, feedback)

    print("RL Updated ✔")

In [ ]:
predict_student([3.2, 1, 1, 6, 7, 5, 1])

predict_student([3.7, 2, 0, 8, 7, 6, 2])

predict_student([3.9, 3, 0, 9, 9, 8, 4])


🎯 Prediction Result
Placement Probability: 0.1
Prediction: NOT PLACED ❌
Expected Salary: 0
Was this correct? (correct/wrong): correct
RL Updated ✔

🎯 Prediction Result
Placement Probability: 0.17
Prediction: NOT PLACED ❌
Expected Salary: 0
Was this correct? (correct/wrong): wrong
RL Updated ✔

🎯 Prediction Result
Placement Probability: 0.53
Prediction: PLACED 🎉
Expected Salary: 11416.95
Was this correct? (correct/wrong): correct
RL Updated ✔
